# Session 2 
#### Iteration 6: Production RAG - final

####  Final RAG deliverable for session 2.   Only using one embedding model OpenAI
Also, including additional context to chunks before embedding.  We already included the document name, and section/page number.  Now, we will also include context provided by the LLM model itself.  Using caching as well to cache the document. 

#### New concepts for this iteration:
 - Simplified to use only OpenAI (text-embedding-3-small) for embedding.
 - Interactive chat (nothing fancy. Print to terminal output, displayed at top of Cursor) and allow for user input. 
 - Log queries to simple csv file for review.  Token budget implemented.

#### Initialization: Load env variables and login to HuggingFace to avoid rate limits

In [ ]:
import os
import re
import tiktoken
import chromadb
import openai as openai_client
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

load_dotenv(find_dotenv())

In [ ]:
# INGEST:  This Python cell loads the PDF files from the data directory and extracts the sections.
data_dir = Path("data/baldwin")

# PyPDFLoader loads each page as a separate LangChain Document with metadata
# (source filename, page number).  We load all PDFs and collect their pages.
raw_docs = []
for pdf_path in sorted(data_dir.glob("*.pdf")):
    loader = PyPDFLoader(str(pdf_path))
    raw_docs.extend(loader.load())

print(f"Loaded {len(raw_docs)} pages from {len(list(data_dir.glob('*.pdf')))} PDFs")

# RecursiveCharacterTextSplitter tries to split on paragraphs first (\n\n),
# then sentences (\n), then words — preserving as much semantic coherence as
# possible while keeping chunks within chunk_size characters.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # about 375 tokens per chunk
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],
)

all_chunks = splitter.split_documents(raw_docs)

#
_HEADING_RE = re.compile(
    r'(?:Chapter\s+[A-Z]?\d+[^\n]*|ARTICLE\s+[IVXLC]+[^\n]*|§\s*[A-Z]?\d+[-\w]*\.[^\n]*)',
    re.MULTILINE,
)

# 
def _extract_heading(text: str, page: int) -> str:
    match = _HEADING_RE.search(text)
    if match:
        return match.group(0).strip()[:80]
    return f"p. {page + 1}"

# Normalise into the same list-of-dicts format the rest of the notebook uses
all_sections = [
    {
        "text": doc.page_content,
        "source": Path(doc.metadata.get("source", "")).name,
        "chunk_index": i,
        "heading": _extract_heading(doc.page_content, doc.metadata.get("page", 0)),
    }
    for i, doc in enumerate(all_chunks)
]

print(f"Split into {len(all_sections)} chunks (avg {sum(len(s['text']) for s in all_sections) // len(all_sections)} chars/chunk)")

#### INDEX: Simplification... we will only embed using the OpenAI embeddings model.  No need to continually build three collections at this point.  

In [ ]:
# INDEXING:  
#### Build the OpenAI collection
import openai as openai_client

OPENAI_MAX_TOKENS = 7_500  # safely under the 8191 token limit for text-embedding-3-small
OPENAI_BATCH_SIZE = 500    # stay well under OpenAI's 2048 inputs-per-request limit
_enc = tiktoken.get_encoding("cl100k_base")

# This function sanitizes the text by removing any null bytes and encoding it to UTF-8.
# It also limits the number of tokens to the maximum number of tokens allowed by the model.
def sanitize(text: str, max_tokens: int | None = None) -> str:
    text = text.replace("\x00", "").encode("utf-8", errors="ignore").decode("utf-8").strip()
    if max_tokens:
        tokens = _enc.encode(text)
        if len(tokens) > max_tokens:
            text = _enc.decode(tokens[:max_tokens])
    return text

# This function embeds the text using the OpenAI API.
# It embeds the text in batches of 500 to stay under the 2048 inputs-per-request limit.
def embed_openai_batched(docs: list[str], model: str = "text-embedding-3-small") -> list[list[float]]:
    client_oa = openai_client.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    embeddings = []
    for i in range(0, len(docs), OPENAI_BATCH_SIZE):
        batch = docs[i:i + OPENAI_BATCH_SIZE]
        response = client_oa.embeddings.create(input=batch, model=model)
        embeddings.extend([r.embedding for r in response.data])
        print(f"  OpenAI: embedded {min(i + OPENAI_BATCH_SIZE, len(docs))}/{len(docs)} docs")
    return embeddings

chroma_client = chromadb.EphemeralClient()
collections = {}

# --- Now create the OpenAI collection, use the OpenAI model (pre-computed embeddings in controlled batches) ---
openai_docs = [sanitize(s["text"], OPENAI_MAX_TOKENS) for s in all_sections]
print("Building OpenAI embeddings...")
openai_embeddings = embed_openai_batched(openai_docs)
col = chroma_client.get_or_create_collection(
    name="text_embedding_3_small_openai",
    embedding_function=OpenAIEmbeddingFunction(api_key=os.environ["OPENAI_API_KEY"], model_name="text-embedding-3-small"),
)
col.add(
    documents=openai_docs,
    embeddings=openai_embeddings,
    metadatas=[{"source": s["source"], "chunk_index": s["chunk_index"], "heading": s["heading"]} for s in all_sections],
    ids=[f"{s['source']}_chunk_{s['chunk_index']}" for s in all_sections],
)
collections["text-embedding-3-small (openai)"] = col
print("Built collection: text-embedding-3-small (openai)")

#### RETRIEVAL INTERACTIVE: This cell presents the user an interactive loop whereby the user can query a collection and answer questions by sending the RAG chunks to the LLM mode for an answer. 
- We are using Anthropic LLM 
- Conversation history is built during the interactive session and sent along with the RAG chunks to the LLM.  The conversation history can be cleared on the fly by using the "clear" command. 
- The user can choose to change the embedding model (and corresponding collection) on the fly by using the "model" command
- The results of the queries are written to a csv file for further analysis with other tools
- There are timeout and token limits that are observed and these are controlled via variables stored in the .env file in main folder

In [ ]:
import csv
import anthropic
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

# --- governance + logging config (set in .env) ---
TOKEN_BUDGET = int(os.environ.get("RAG_TOKEN_BUDGET", 50_000))
TIMEOUT_SECS = int(os.environ.get("RAG_TIMEOUT_SECS", 30))
LOG_FILE     = Path(os.environ.get("RAG_LOG_FILE", "rag_session_log.csv"))

def _log_turn(query: str, chunks: list, distances: list, answer: str, tokens: int):
    """Append one row per query to the CSV log file."""
    is_new = not LOG_FILE.exists()
    n = len(chunks)
    chunk_headers = [f"chunk_{i+1}_{f}" for i in range(n) for f in ("score", "source", "heading", "text")]
    fieldnames = ["timestamp", "query", "answer", "tokens_turn"] + chunk_headers

    row = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "query":     query,
        "answer":    answer,
        "tokens_turn": tokens,
    }
    for i, ((text, meta), dist) in enumerate(zip(chunks, distances)):
        prefix = f"chunk_{i+1}_"
        row[prefix + "score"]        = round(1 - dist, 4)
        row[prefix + "source"]       = meta["source"]
        row[prefix + "heading"]      = meta["heading"]
        row[prefix + "text"] = text.replace("\n", " ")

    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if is_new:
            writer.writeheader()
        writer.writerow(row)
    print(f"[Logged to {LOG_FILE}]")

# Conversation memory — grows with each turn and is sent as context on every request.
# Inspect at any time, or reset with: conversation_history.clear()
conversation_history: list[dict] = []
session_tokens_used: int = 0

def compare_interactive(
    collection_name: str = "text-embedding-3-small (openai)",
    n_results: int = 7,
):
    """
    Interactive RAG chat session with conversation memory and governance.

    Governance:
      - TOKEN_BUDGET (RAG_TOKEN_BUDGET in .env): hard cap on total tokens per session
      - TIMEOUT_SECS (RAG_TIMEOUT_SECS in .env): per-request timeout in seconds

    Commands during the session:
      quit / exit / q  — end the session
      clear            — wipe conversation memory and reset token counter
    """
    global conversation_history, session_tokens_used

    col = collections[collection_name]
    ac = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    def _print_header():
        print(f"\nRAG Chat  |  active model: '{collection_name}'")
        print(f"Governance: token budget={TOKEN_BUDGET:,}  |  timeout={TIMEOUT_SECS}s")
        print("Commands: 'quit' to exit  |  'clear' to reset memory  |  'model' to switch embedding model\n")

    _print_header()

    while True:
        # --- enforce token budget before accepting input ---
        remaining = TOKEN_BUDGET - session_tokens_used
        if remaining <= 0:
            print(f"[Session token budget of {TOKEN_BUDGET:,} exhausted. Type 'clear' to reset or 'quit' to exit.]")

        query = input("You: ").strip()

        if not query:
            continue
        if query.lower() in ("quit", "exit", "q"):
            print(f"Session ended. Tokens used this session: {session_tokens_used:,}")
            break
        if query.lower() == "clear":
            conversation_history.clear()
            session_tokens_used = 0
            print("[Memory cleared. Token counter reset.]\n")
            continue
        if session_tokens_used >= TOKEN_BUDGET:
            print("[Cannot process — token budget exhausted. Type 'clear' to reset.]\n")
            continue

        # --- retrieve relevant chunks ---
        results = col.query(query_texts=[query], n_results=n_results, include=["documents", "metadatas", "distances"])
        context_chunks = list(zip(results["documents"][0], results["metadatas"][0]))
        distances      = results["distances"][0]
        context_str = "\n\n".join(
            f"[{i+1}] {meta['source']} | {meta['heading']}\n{text}"
            for i, (text, meta) in enumerate(context_chunks)
        )

        # --- build message list from growing history ---
        messages = []
        for turn in conversation_history:
            messages.append({"role": "user",      "content": turn["question"]})
            messages.append({"role": "assistant",  "content": turn["answer"]})
        messages.append({"role": "user", "content": query})

        system_prompt = (
            "You are a precise assistant answering questions about Baldwin Borough "
            "municipal regulations. You must follow these rules strictly:\n\n"
            "1. Answer ONLY from the numbered context passages below. "
            "Do NOT use your training data or general knowledge.\n"
            "2. Every factual claim MUST be followed immediately by an inline "
            "citation in the form [N] referencing the passage number it came from.\n"
            "3. If a question cannot be answered from the context, say exactly: "
            "'This information is not available in the provided documents.'\n"
            "4. Never speculate or infer beyond what the passages explicitly state.\n\n"
            f"CONTEXT PASSAGES:\n{context_str}"
        )

        # --- call Claude with hard timeout ---
        try:
            with ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(
                    ac.messages.create,
                    model="claude-sonnet-4-5",
                    max_tokens=1024,
                    system=system_prompt,
                    messages=messages,
                )
                response = future.result(timeout=TIMEOUT_SECS)
        except FuturesTimeoutError:
            print(f"\n[Request timed out after {TIMEOUT_SECS}s. Please try again.]\n")
            continue
        except Exception as e:
            print(f"\n[API error: {e}]\n")
            continue

        # --- track token usage ---
        turn_tokens = response.usage.input_tokens + response.usage.output_tokens
        session_tokens_used += turn_tokens
        remaining_after = TOKEN_BUDGET - session_tokens_used

        answer = response.content[0].text
        print(f"\n{'─' * 60}")
        print(f"You asked: {query}")
        print(f"{'─' * 60}")
        print(f"\nAssistant: {answer}")

        # --- numbered citation index with full chunk text ---
        print("\n" + "─" * 60)
        print("Retrieved passages (cited as [N] in the answer above):")
        for i, (text, meta) in enumerate(context_chunks):
            print(f"\n  [{i+1}] {meta['source']} | {meta['heading']}  (score: {round(1 - distances[i], 4)})")
            print(f"  {text}")
        print("\n" + "─" * 60)
        print(f"[Tokens this turn: {turn_tokens:,} | Session total: {session_tokens_used:,} | Budget remaining: {remaining_after:,}]\n")

        conversation_history.append({"question": query, "answer": answer})
        _log_turn(query, context_chunks, distances, answer, turn_tokens)


# Run the session — re-execute this cell to continue (memory + token count persist).
# Call conversation_history.clear() and reset session_tokens_used = 0 to start fresh.
compare_interactive()